# <center>Lab 8 — Widoki i widoki systemowe PostgreSQL</center>

Tym razem skupiamy się na **widokach użytkownika** oraz **widokach systemowych PostgreSQL**, które pomagają analizować strukturę tabel, statystyki i wydajność.

Ćwiczenie jest przygotowane dla PostgreSQL i pgAdmin Query Tool.

## 1. Cele ćwiczenia

Po wykonaniu ćwiczenia student powinien umieć:

- utworzyć zwykły widok `VIEW`,
- utworzyć widok zmaterializowany `MATERIALIZED VIEW`,
- rozumieć schematy jako przestrzenie nazw i odwoływać się do obiektów przez `schema.object`,
- analizować interakcje między schematami przez widoki, klucze obce i `search_path`,
- analizować strukturę tabel przez `information_schema`,
- sprawdzać definicje widoków i indeksów przez `pg_catalog`,
- korzystać z `pg_stat_user_tables`, `pg_stat_user_indexes`, `pg_stats` i `pg_stat_activity`,
- porównać plany zapytań przez `EXPLAIN (ANALYZE, BUFFERS)`,
- sprawdzić wpływ indeksów, `ANALYZE` i `VACUUM (ANALYZE)` na zachowanie bazy.

## 2. Widok w SQL

**Widok** to zapisane zapytanie SQL, które można odpytywać podobnie jak tabelę. Widok zwykły nie przechowuje osobnej kopii danych; przy odczycie PostgreSQL wykonuje zapytanie na tabelach źródłowych.

```sql
CREATE VIEW nazwa_widoku AS
SELECT ...
FROM ...;
```

Widoki są przydatne do uproszczenia zapytań, ukrycia złożonych `JOIN`, budowania warstwy raportowej i ograniczania widoczności wybranych kolumn.

## 3. Widok zmaterializowany

**Widok zmaterializowany** przechowuje wynik zapytania fizycznie. Odczyt może być szybszy, ale dane nie odświeżają się automatycznie po zmianach w tabelach źródłowych.

```sql
REFRESH MATERIALIZED VIEW nazwa_widoku;
```

Dla raportów okresowych widok zmaterializowany jest często dobrym kompromisem między szybkością odczytu a aktualnością danych.

## 4. Baza treningowa

Model przedstawia uproszczony sklep internetowy:

- `customers` — klienci,
- `products` — produkty,
- `orders` — zamówienia,
- `order_items` — pozycje zamówień.

Relacje zawierają reguły `ON UPDATE` i `ON DELETE`, aby później można było analizować je przez widoki systemowe.

In [1]:
DROP SCHEMA IF EXISTS lab5_reporting CASCADE;
DROP SCHEMA IF EXISTS lab5_archive CASCADE;
DROP SCHEMA IF EXISTS lab5 CASCADE;
CREATE SCHEMA lab5;
SET search_path TO lab5, public;

SyntaxError: invalid syntax (2005335983.py, line 1)

In [ ]:
CREATE TABLE lab5.customers (
    customer_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    email TEXT NOT NULL UNIQUE,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    city TEXT NOT NULL,
    created_at TIMESTAMP NOT NULL DEFAULT now()
);

CREATE TABLE lab5.products (
    product_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    sku TEXT NOT NULL UNIQUE,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    price NUMERIC(10, 2) NOT NULL CHECK (price >= 0),
    active BOOLEAN NOT NULL DEFAULT true
);

CREATE TABLE lab5.orders (
    order_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_id BIGINT NOT NULL REFERENCES lab5.customers(customer_id)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,
    order_status TEXT NOT NULL CHECK (order_status IN ('new', 'paid', 'shipped', 'cancelled')),
    order_date TIMESTAMP NOT NULL DEFAULT now()
);

CREATE TABLE lab5.order_items (
    order_item_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    order_id BIGINT NOT NULL REFERENCES lab5.orders(order_id)
        ON UPDATE CASCADE
        ON DELETE CASCADE,
    product_id BIGINT NOT NULL REFERENCES lab5.products(product_id)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,
    quantity INTEGER NOT NULL CHECK (quantity > 0),
    unit_price NUMERIC(10, 2) NOT NULL CHECK (unit_price >= 0)
);

## 5. Dane testowe

Dane generujemy funkcją `generate_series`, aby zapytania miały wystarczającą liczbę rekordów do obserwacji planów wykonania.

In [ ]:
INSERT INTO lab5.customers(email, first_name, last_name, city, created_at)
SELECT
    'user' || g || '@example.com',
    'Name' || g,
    'Surname' || g,
    (ARRAY['Warszawa', 'Kraków', 'Gdańsk', 'Poznań', 'Wrocław', 'Łódź'])[1 + (g % 6)],
    now() - ((g % 365) || ' days')::interval
FROM generate_series(1, 20000) AS g;

INSERT INTO lab5.products(sku, product_name, category, price, active)
SELECT
    'SKU-' || g,
    'Product ' || g,
    (ARRAY['book', 'electronics', 'home', 'sport', 'beauty'])[1 + (g % 5)],
    round((10 + random() * 990)::numeric, 2),
    g % 20 <> 0
FROM generate_series(1, 1000) AS g;

INSERT INTO lab5.orders(customer_id, order_status, order_date)
SELECT
    1 + (random() * 19999)::int,
    (ARRAY['new', 'paid', 'shipped', 'cancelled'])[1 + (random() * 3)::int],
    now() - ((random() * 180)::int || ' days')::interval
FROM generate_series(1, 80000) AS g;

INSERT INTO lab5.order_items(order_id, product_id, quantity, unit_price)
SELECT
    1 + (random() * 79999)::int,
    1 + (random() * 999)::int,
    1 + (random() * 4)::int,
    p.price
FROM generate_series(1, 220000) AS g
JOIN LATERAL (
    SELECT price
    FROM lab5.products
    WHERE product_id = 1 + (random() * 999)::int
    LIMIT 1
) AS p ON true;

ANALYZE lab5.customers;
ANALYZE lab5.products;
ANALYZE lab5.orders;
ANALYZE lab5.order_items;

## 6. Schematy w bazie danych i interakcje między nimi

**Schemat** w PostgreSQL jest przestrzenią nazw wewnątrz jednej bazy danych. Dzięki schematom można rozdzielać obiekty według odpowiedzialności, np. dane transakcyjne, raportowanie, archiwum albo moduły aplikacji.

W tym ćwiczeniu wykorzystamy trzy schematy:

- `lab5` — podstawowe dane transakcyjne,
- `lab5_reporting` — widoki przeznaczone do raportowania,
- `lab5_archive` — dane pomocnicze lub audytowe.

Najważniejsze formy interakcji między schematami:

1. odwołanie przez pełną nazwę `schema.object`,
2. `JOIN` między tabelami z różnych schematów,
3. widok w jednym schemacie oparty na tabelach z innego schematu,
4. klucz obcy wskazujący tabelę w innym schemacie,
5. `search_path`, czyli kolejność wyszukiwania nazw bez prefiksu schematu,
6. uprawnienia nadawane osobno do schematu i do obiektów w schemacie.


In [ ]:
CREATE SCHEMA IF NOT EXISTS lab5_reporting;
CREATE SCHEMA IF NOT EXISTS lab5_archive;

DROP VIEW IF EXISTS lab5_reporting.v_status_changes;
DROP VIEW IF EXISTS lab5_reporting.v_orders_with_customer;
DROP TABLE IF EXISTS lab5_archive.order_status_log;

CREATE TABLE lab5_archive.order_status_log (
    log_id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    order_id BIGINT NOT NULL REFERENCES lab5.orders(order_id)
        ON UPDATE CASCADE
        ON DELETE CASCADE,
    old_status TEXT,
    new_status TEXT NOT NULL,
    changed_at TIMESTAMP NOT NULL DEFAULT now()
);

INSERT INTO lab5_archive.order_status_log(order_id, old_status, new_status)
SELECT order_id, NULL, order_status
FROM lab5.orders
WHERE order_id <= 10;

CREATE OR REPLACE VIEW lab5_reporting.v_orders_with_customer AS
SELECT
    o.order_id,
    o.order_date,
    o.order_status,
    c.customer_id,
    c.email,
    c.city,
    count(oi.order_item_id) AS lines_count,
    coalesce(sum(oi.quantity * oi.unit_price), 0) AS order_value
FROM lab5.orders AS o
JOIN lab5.customers AS c ON c.customer_id = o.customer_id
LEFT JOIN lab5.order_items AS oi ON oi.order_id = o.order_id
GROUP BY o.order_id, o.order_date, o.order_status, c.customer_id, c.email, c.city;

CREATE OR REPLACE VIEW lab5_reporting.v_status_changes AS
SELECT
    l.log_id,
    l.changed_at,
    o.order_id,
    c.email,
    l.old_status,
    l.new_status
FROM lab5_archive.order_status_log AS l
JOIN lab5.orders AS o ON o.order_id = l.order_id
JOIN lab5.customers AS c ON c.customer_id = o.customer_id;


### Sprawdzenie obiektów i działania `search_path`

W poniższych zapytaniach widać, że jeden raportowy widok może pobierać dane z tabel w schemacie `lab5`, a drugi łączy dane z `lab5`, `lab5_archive` i `lab5_reporting`. Zmiana `search_path` pozwala odwołać się do widoku bez prefiksu schematu, ale w kodzie produkcyjnym bezpieczniej jest używać pełnych nazw tam, gdzie może pojawić się niejednoznaczność.


In [ ]:
SELECT schema_name
FROM information_schema.schemata
WHERE schema_name LIKE 'lab5%'
ORDER BY schema_name;

SELECT
    n.nspname AS schema_name,
    c.relname AS object_name,
    CASE c.relkind
        WHEN 'r' THEN 'table'
        WHEN 'v' THEN 'view'
        WHEN 'm' THEN 'materialized view'
        WHEN 'i' THEN 'index'
        WHEN 'S' THEN 'sequence'
        ELSE c.relkind::text
    END AS object_type
FROM pg_catalog.pg_class AS c
JOIN pg_catalog.pg_namespace AS n ON n.oid = c.relnamespace
WHERE n.nspname IN ('lab5', 'lab5_reporting', 'lab5_archive')
ORDER BY schema_name, object_type, object_name;

SELECT *
FROM lab5_reporting.v_orders_with_customer
ORDER BY order_id
LIMIT 10;

SET search_path TO lab5_reporting, lab5, public;

SELECT *
FROM v_orders_with_customer
ORDER BY order_id
LIMIT 5;

SET search_path TO lab5, public;


### Uprawnienia między schematami

W praktyce bardzo często tworzy się schemat raportowy, do którego użytkownicy mają tylko prawo odczytu, bez bezpośredniego dostępu do tabel źródłowych. Przykład zostawiamy jako komentarz, ponieważ w lokalnym środowisku rola może nie istnieć.


In [ ]:
-- CREATE ROLE readonly_reporter LOGIN PASSWORD 'change_me';
-- GRANT USAGE ON SCHEMA lab5_reporting TO readonly_reporter;
-- GRANT SELECT ON ALL TABLES IN SCHEMA lab5_reporting TO readonly_reporter;
-- ALTER DEFAULT PRIVILEGES IN SCHEMA lab5_reporting
--     GRANT SELECT ON TABLES TO readonly_reporter;


## 7. Zwykłe widoki analityczne

Pierwszy widok łączy dane z czterech tabel i tworzy wygodną warstwę raportową.

In [ ]:
CREATE OR REPLACE VIEW lab5.v_order_details AS
SELECT
    o.order_id,
    o.order_date,
    o.order_status,
    c.customer_id,
    c.email,
    c.city,
    oi.order_item_id,
    p.product_id,
    p.product_name,
    p.category,
    oi.quantity,
    oi.unit_price,
    oi.quantity * oi.unit_price AS line_total
FROM lab5.orders AS o
JOIN lab5.customers AS c ON c.customer_id = o.customer_id
JOIN lab5.order_items AS oi ON oi.order_id = o.order_id
JOIN lab5.products AS p ON p.product_id = oi.product_id;

CREATE OR REPLACE VIEW lab5.v_sales_by_city AS
SELECT city, count(DISTINCT order_id) AS orders_count, count(*) AS lines_count, sum(line_total) AS revenue
FROM lab5.v_order_details
GROUP BY city;

CREATE OR REPLACE VIEW lab5.v_sales_by_category AS
SELECT category, count(DISTINCT order_id) AS orders_count, sum(quantity) AS sold_units, sum(line_total) AS revenue
FROM lab5.v_order_details
GROUP BY category;

## 8. Widok zmaterializowany

Widok zmaterializowany może przyspieszyć raporty agregujące dane. Po zmianach w danych wymaga odświeżenia.

In [ ]:
DROP MATERIALIZED VIEW IF EXISTS lab5.mv_daily_sales;

CREATE MATERIALIZED VIEW lab5.mv_daily_sales AS
SELECT
    date_trunc('day', order_date)::date AS sales_day,
    category,
    count(DISTINCT order_id) AS orders_count,
    sum(quantity) AS sold_units,
    sum(line_total) AS revenue
FROM lab5.v_order_details
GROUP BY date_trunc('day', order_date)::date, category;

CREATE UNIQUE INDEX mv_daily_sales_uq
    ON lab5.mv_daily_sales(sales_day, category);

-- REFRESH MATERIALIZED VIEW lab5.mv_daily_sales;
-- REFRESH MATERIALIZED VIEW CONCURRENTLY lab5.mv_daily_sales;

## 9. `information_schema` — analiza struktury

`information_schema` zawiera widoki opisujące obiekty w aktualnej bazie. Jest bardziej przenośne między systemami SQL niż `pg_catalog`, ale nie pokazuje wszystkich szczegółów specyficznych dla PostgreSQL.

In [ ]:
SELECT table_schema, table_name, table_type
FROM information_schema.tables
WHERE table_schema = 'lab5'
ORDER BY table_name;

In [ ]:
SELECT table_name, ordinal_position, column_name, data_type, is_nullable, column_default
FROM information_schema.columns
WHERE table_schema = 'lab5'
ORDER BY table_name, ordinal_position;

In [ ]:
SELECT
    tc.table_name,
    tc.constraint_name,
    tc.constraint_type,
    kcu.column_name,
    ccu.table_name AS foreign_table_name,
    ccu.column_name AS foreign_column_name
FROM information_schema.table_constraints AS tc
LEFT JOIN information_schema.key_column_usage AS kcu
    ON tc.constraint_name = kcu.constraint_name
   AND tc.table_schema = kcu.table_schema
LEFT JOIN information_schema.constraint_column_usage AS ccu
    ON tc.constraint_name = ccu.constraint_name
   AND tc.table_schema = ccu.table_schema
WHERE tc.table_schema = 'lab5'
ORDER BY tc.table_name, tc.constraint_type, tc.constraint_name;

## 10. Reguły `ON UPDATE` i `ON DELETE`

Sama informacja o kluczu obcym nie wystarczy. W analizie struktury ważne jest także to, czy baza blokuje usunięcie rekordu nadrzędnego, ustawia `NULL`, czy usuwa zależne rekordy kaskadowo.

In [ ]:
SELECT
    rc.constraint_name,
    kcu.table_name,
    kcu.column_name,
    ccu.table_name AS referenced_table,
    ccu.column_name AS referenced_column,
    rc.update_rule,
    rc.delete_rule
FROM information_schema.referential_constraints AS rc
JOIN information_schema.key_column_usage AS kcu
    ON rc.constraint_name = kcu.constraint_name
   AND rc.constraint_schema = kcu.constraint_schema
JOIN information_schema.constraint_column_usage AS ccu
    ON rc.unique_constraint_name = ccu.constraint_name
   AND rc.unique_constraint_schema = ccu.constraint_schema
WHERE rc.constraint_schema = 'lab5'
ORDER BY kcu.table_name, rc.constraint_name;

## 11. `pg_catalog` — widoki specyficzne dla PostgreSQL

Do analizy obiektów PostgreSQL przydatne są m.in. `pg_views`, `pg_indexes`, `pg_class`, `pg_namespace`, `pg_constraint` i `pg_attribute`.

In [ ]:
SELECT schemaname, viewname, definition
FROM pg_catalog.pg_views
WHERE schemaname = 'lab5'
ORDER BY viewname;

In [ ]:
SELECT schemaname, tablename, indexname, indexdef
FROM pg_catalog.pg_indexes
WHERE schemaname = 'lab5'
ORDER BY tablename, indexname;

In [ ]:
SELECT
    n.nspname AS schema_name,
    c.relname AS object_name,
    c.relkind,
    CASE c.relkind
        WHEN 'r' THEN 'table'
        WHEN 'i' THEN 'index'
        WHEN 'v' THEN 'view'
        WHEN 'm' THEN 'materialized view'
        WHEN 'S' THEN 'sequence'
        ELSE c.relkind::text
    END AS object_type,
    pg_size_pretty(pg_total_relation_size(c.oid)) AS total_size
FROM pg_catalog.pg_class AS c
JOIN pg_catalog.pg_namespace AS n ON n.oid = c.relnamespace
WHERE n.nspname = 'lab5'
ORDER BY pg_total_relation_size(c.oid) DESC;

## 12. Widoki statystyczne

PostgreSQL udostępnia widoki statystyczne, które pozwalają sprawdzić m.in. liczbę skanów sekwencyjnych, skanów indeksowych, operacji DML, martwych krotek oraz czas ostatniego `VACUUM` i `ANALYZE`.

In [ ]:
SELECT
    schemaname,
    relname,
    seq_scan,
    idx_scan,
    n_tup_ins,
    n_tup_upd,
    n_tup_del,
    n_dead_tup,
    last_vacuum,
    last_autovacuum,
    last_analyze,
    last_autoanalyze
FROM pg_stat_user_tables
WHERE schemaname = 'lab5'
ORDER BY relname;

In [ ]:
SELECT
    schemaname,
    relname AS table_name,
    indexrelname AS index_name,
    idx_scan,
    idx_tup_read,
    idx_tup_fetch
FROM pg_stat_user_indexes
WHERE schemaname = 'lab5'
ORDER BY idx_scan DESC, indexrelname;

In [ ]:
SELECT
    schemaname,
    relname AS table_name,
    pg_size_pretty(pg_relation_size(relid)) AS table_size,
    pg_size_pretty(pg_indexes_size(relid)) AS indexes_size,
    pg_size_pretty(pg_total_relation_size(relid)) AS total_size
FROM pg_stat_user_tables
WHERE schemaname = 'lab5'
ORDER BY pg_total_relation_size(relid) DESC;

## 13. `pg_stats` i statystyki planera

`pg_stats` pokazuje statystyki kolumn wykorzystywane przez planner zapytań. Są one odświeżane przez `ANALYZE` oraz `VACUUM ANALYZE`.

In [ ]:
SELECT
    schemaname,
    tablename,
    attname,
    null_frac,
    n_distinct,
    most_common_vals,
    most_common_freqs
FROM pg_stats
WHERE schemaname = 'lab5'
  AND tablename IN ('orders', 'order_items', 'products')
ORDER BY tablename, attname;

## 14. Aktywne sesje i zapytania

`pg_stat_activity` pomaga diagnozować długo działające zapytania, blokady i sesje oczekujące na zasoby.

In [ ]:
SELECT
    datname,
    pid,
    usename,
    state,
    wait_event_type,
    wait_event,
    now() - query_start AS query_duration,
    left(query, 120) AS query_preview
FROM pg_stat_activity
WHERE datname = current_database()
ORDER BY query_start NULLS LAST;

## 15. `EXPLAIN (ANALYZE, BUFFERS)`

`EXPLAIN` pokazuje plan zapytania. `ANALYZE` wykonuje zapytanie i pokazuje rzeczywiste czasy, a `BUFFERS` dodaje informacje o pracy z buforami. Dzięki temu można porównać koszt zapytania przed i po dodaniu indeksów.

In [ ]:
EXPLAIN (ANALYZE, BUFFERS)
SELECT o.order_status, count(*) AS orders_count
FROM lab5.orders AS o
WHERE o.order_status = 'paid'
  AND o.order_date >= now() - interval '30 days'
GROUP BY o.order_status;

EXPLAIN (ANALYZE, BUFFERS)
SELECT p.category, sum(oi.quantity * oi.unit_price) AS revenue
FROM lab5.order_items AS oi
JOIN lab5.products AS p ON p.product_id = oi.product_id
JOIN lab5.orders AS o ON o.order_id = oi.order_id
WHERE o.order_date >= now() - interval '30 days'
GROUP BY p.category
ORDER BY revenue DESC;

## 16. Dodanie indeksów i ponowny pomiar

Indeksy mogą przyspieszyć odczyt, ale zwiększają koszt zapisu i zajmują miejsce na dysku. Po utworzeniu indeksów warto uruchomić `ANALYZE`.

In [ ]:
CREATE INDEX IF NOT EXISTS idx_orders_status_date ON lab5.orders(order_status, order_date);
CREATE INDEX IF NOT EXISTS idx_orders_order_date ON lab5.orders(order_date);
CREATE INDEX IF NOT EXISTS idx_order_items_order_id ON lab5.order_items(order_id);
CREATE INDEX IF NOT EXISTS idx_order_items_product_id ON lab5.order_items(product_id);
CREATE INDEX IF NOT EXISTS idx_products_category ON lab5.products(category);

ANALYZE lab5.orders;
ANALYZE lab5.order_items;
ANALYZE lab5.products;

In [ ]:
EXPLAIN (ANALYZE, BUFFERS)
SELECT o.order_status, count(*) AS orders_count
FROM lab5.orders AS o
WHERE o.order_status = 'paid'
  AND o.order_date >= now() - interval '30 days'
GROUP BY o.order_status;

EXPLAIN (ANALYZE, BUFFERS)
SELECT p.category, sum(oi.quantity * oi.unit_price) AS revenue
FROM lab5.order_items AS oi
JOIN lab5.products AS p ON p.product_id = oi.product_id
JOIN lab5.orders AS o ON o.order_id = oi.order_id
WHERE o.order_date >= now() - interval '30 days'
GROUP BY p.category
ORDER BY revenue DESC;

## 17. `UPDATE`, `DELETE`, martwe krotki i `VACUUM`

Po `UPDATE` i `DELETE` mogą pozostać martwe krotki. `VACUUM` sprząta martwe wersje wierszy, a `ANALYZE` odświeża statystyki planera.

In [ ]:
UPDATE lab5.orders
SET order_status = 'cancelled'
WHERE order_status = 'new'
  AND order_id % 10 = 0;

DELETE FROM lab5.order_items
WHERE order_item_id % 50 = 0;

SELECT schemaname, relname, n_tup_upd, n_tup_del, n_dead_tup,
       last_vacuum, last_autovacuum, last_analyze, last_autoanalyze
FROM pg_stat_user_tables
WHERE schemaname = 'lab5'
ORDER BY n_dead_tup DESC;

VACUUM (ANALYZE) lab5.orders;
VACUUM (ANALYZE) lab5.order_items;

SELECT schemaname, relname, n_dead_tup, last_vacuum, last_analyze
FROM pg_stat_user_tables
WHERE schemaname = 'lab5'
ORDER BY relname;

## 18. Dodatek: `pg_stat_statements`

`pg_stat_statements` jest rozszerzeniem do zbierania statystyk planowania i wykonania zapytań SQL. Pozwala znaleźć zapytania o największym łącznym albo średnim czasie wykonania. W wielu instalacjach wymaga ustawienia `shared_preload_libraries` oraz restartu serwera.

In [ ]:
CREATE EXTENSION IF NOT EXISTS pg_stat_statements;

SELECT
    calls,
    round(total_exec_time::numeric, 2) AS total_exec_time_ms,
    round(mean_exec_time::numeric, 2) AS mean_exec_time_ms,
    rows,
    left(query, 120) AS query_preview
FROM pg_stat_statements
WHERE dbid = (SELECT oid FROM pg_database WHERE datname = current_database())
ORDER BY total_exec_time DESC
LIMIT 10;

-- SELECT pg_stat_statements_reset();

## 19. Różnice względem MySQL

MySQL również ma `information_schema`, ale w analizie wydajności częściej korzysta się z `performance_schema` i schematu `sys`. Odpowiednikiem części pracy z odświeżaniem statystyk jest `ANALYZE TABLE`. PostgreSQL ma rozbudowany zestaw widoków `pg_stat_*` oraz katalog `pg_catalog`.

## 20. Zadania 

1. Utwórz widok `lab5.v_customer_summary` pokazujący klienta, liczbę zamówień, łączną wartość i datę ostatniego zamówienia.
2. Utwórz widok zmaterializowany `lab5.mv_product_sales` z podsumowaniem sprzedaży produktów i dodaj unikalny indeks.
3. Napisz zapytanie do `information_schema`, które znajdzie wszystkie kolumny typu `text` w schemacie `lab5`.
4. Pokaż wszystkie klucze obce wraz z regułami `ON UPDATE` i `ON DELETE`.
5. Porównaj `EXPLAIN (ANALYZE, BUFFERS)` dla `SELECT * FROM lab5.orders WHERE customer_id = 100;` przed i po dodaniu indeksu na `orders(customer_id)`.
6. Wykonaj większe `UPDATE` albo `DELETE`, sprawdź `pg_stat_user_tables` przed i po `VACUUM (ANALYZE)`.
7. Utwórz schemat `lab5_sandbox` i tabelę `lab5_sandbox.order_notes` z kluczem obcym do `lab5.orders`.
8. Utwórz widok w `lab5_reporting`, który łączy dane z `lab5.orders`, `lab5.customers` i `lab5_sandbox.order_notes`.
9. Zmień `search_path` i sprawdź, jak działa zapytanie do widoku bez prefiksu schematu.
10. Opcjonalnie uruchom `pg_stat_statements` i znajdź 10 zapytań o największym średnim czasie wykonania.
